In [3]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import torch
import open_clip
from PIL import Image
# ============================================================
# SETTINGS
# ============================================================
INPUT_DIR = Path("Places_Konkle_Sample300")
FILTERED_DIR = Path("Places_Konkle_No_Humans")
OUTPUT_DIR = Path("clip_rn50_results")
DATASETS = ["places", "konkle"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
N_SELECT = 30
BATCH_SIZE = 32
# Lower = more conservative: more images will be classified as containing humans.
# 0.40 is a  conservative threshold value.
HUMAN_PROB_THRESHOLD = 0.40
# ============================================================
# DEVICE
# ============================================================
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")
# ============================================================
# LOAD OPENCLIP RN50
# ============================================================
MODEL_NAME = "RN50"
PRETRAINED = "openai"
model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model = model.to(DEVICE)
model.eval()
print("OpenCLIP RN50 loaded.")
# ============================================================
# HUMAN / NON-HUMAN PROMPTS
# ============================================================
HUMAN_PROMPTS = [
    "a photo of a person",
    "a photo containing a human",
    "a photo containing people",
    "a photo of a human face",
    "a photo containing a face",
    "a photo of a human hand",
    "a photo containing a hand",
    "a photo of a human arm",
    "a photo containing an arm",
    "a photo of a human leg",
    "a photo containing a leg",
    "a photo of a human foot",
    "a photo containing a foot",
    "a photo of a human body",
    "a photo containing part of a human body",
    "a photo containing visible human skin"
]
NO_HUMAN_PROMPTS = [
    "a photo with no people",
    "a photo containing no humans",
    "an empty scene without people",
    "a scene containing only objects",
    "a scene containing only buildings, objects, or nature",
    "a photo without any human body parts"
]
# ============================================================
# TEXT EMBEDDINGS FOR HUMAN FILTER
# ============================================================
@torch.no_grad()
def encode_text_prompts(prompts):
    tokens = tokenizer(prompts).to(DEVICE)
    features = model.encode_text(tokens)
    features = features / features.norm(dim=-1, keepdim=True)
    return features
human_text_features = encode_text_prompts(HUMAN_PROMPTS)
no_human_text_features = encode_text_prompts(NO_HUMAN_PROMPTS)
# ============================================================
# IMAGE HELPERS
# ============================================================
def get_images(category_dir):
    return sorted([
        path for path in Path(category_dir).iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ])
def load_image(path):
    with Image.open(path) as img:
        return preprocess(img.convert("RGB"))
# ============================================================
# HUMAN DETECTION
# ============================================================
@torch.no_grad()
def classify_human_batch(image_paths):
    valid_paths = []
    tensors = []
    for path in image_paths:
        try:
            tensors.append(load_image(path))
            valid_paths.append(path)
        except Exception as e:
            print(f"Could not load {path}: {e}")
    if not tensors:
        return []
    images = torch.stack(tensors).to(DEVICE)
    image_features = model.encode_image(images)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    human_similarity = image_features @ human_text_features.T
    no_human_similarity = image_features @ no_human_text_features.T
    human_score = human_similarity.max(dim=1).values
    no_human_score = no_human_similarity.max(dim=1).values
    logits = torch.stack([human_score, no_human_score], dim=1) * 100
    probabilities = logits.softmax(dim=1)
    human_probability = probabilities[:, 0]
    results = []
    for path, h_score, nh_score, h_prob in zip(
        valid_paths,
        human_score.cpu().numpy(),
        no_human_score.cpu().numpy(),
        human_probability.cpu().numpy()
    ):
        contains_human = float(h_prob) >= HUMAN_PROB_THRESHOLD
        results.append({
            "image_path": str(path),
            "human_similarity": float(h_score),
            "no_human_similarity": float(nh_score),
            "human_probability": float(h_prob),
            "contains_human": bool(contains_human)
        })
    return results
# ============================================================
# FILTER ONE DATASET
# ============================================================
def filter_dataset(dataset_name):
    source_dataset = INPUT_DIR / dataset_name
    destination_dataset = FILTERED_DIR / dataset_name
    if not source_dataset.exists():
        print(f"Dataset not found: {source_dataset}")
        return pd.DataFrame()
    category_dirs = sorted([
        path for path in source_dataset.iterdir()
        if path.is_dir()
    ])
    records = []
    print(f"\nFiltering {dataset_name}: {len(category_dirs)} categories")
    for category_dir in category_dirs:
        image_paths = get_images(category_dir)
        output_category = destination_dataset / category_dir.name
        output_category.mkdir(parents=True, exist_ok=True)
        kept = 0
        removed = 0
        print(f"{dataset_name}/{category_dir.name}: {len(image_paths)} images")
        for start in range(0, len(image_paths), BATCH_SIZE):
            batch_paths = image_paths[start:start + BATCH_SIZE]
            batch_results = classify_human_batch(batch_paths)
            for result in batch_results:
                image_path = Path(result["image_path"])
                if result["contains_human"]:
                    removed += 1
                    decision = "excluded"
                else:
                    shutil.copy2(image_path, output_category / image_path.name)
                    kept += 1
                    decision = "kept"
                records.append({
                    "dataset": dataset_name,
                    "category": category_dir.name,
                    "filename": image_path.name,
                    "image_path": str(image_path),
                    "human_similarity": result["human_similarity"],
                    "no_human_similarity": result["no_human_similarity"],
                    "human_probability": result["human_probability"],
                    "decision": decision
                })
        print(f"  kept={kept}, excluded={removed}")
    return pd.DataFrame(records)
# ============================================================
# RUN HUMAN FILTER
# ============================================================
def run_human_filter():
    FILTERED_DIR.mkdir(parents=True, exist_ok=True)
    frames = []
    for dataset in DATASETS:
        df = filter_dataset(dataset)
        if not df.empty:
            frames.append(df)
    if frames:
        results = pd.concat(frames, ignore_index=True)
        results.to_csv(FILTERED_DIR / "human_filter_results.csv", index=False)
        print(f"\nHuman-filter results saved to {FILTERED_DIR / 'human_filter_results.csv'}")
        print(f"Kept: {(results['decision'] == 'kept').sum()}")
        print(f"Excluded: {(results['decision'] == 'excluded').sum()}")
        return results
    return pd.DataFrame()
# ============================================================
# EXTRACT RN50 IMAGE EMBEDDINGS
# ============================================================
@torch.no_grad()
def extract_image_features(image_paths):
    all_features = []
    valid_paths = []
    for start in range(0, len(image_paths), BATCH_SIZE):
        batch_paths = image_paths[start:start + BATCH_SIZE]
        batch_tensors = []
        batch_valid_paths = []
        for path in batch_paths:
            try:
                batch_tensors.append(load_image(path))
                batch_valid_paths.append(path)
            except Exception as e:
                print(f"Could not load {path}: {e}")
        if not batch_tensors:
            continue
        images = torch.stack(batch_tensors).to(DEVICE)
        features = model.encode_image(images)
        features = features / features.norm(dim=-1, keepdim=True)
        all_features.append(features.cpu().numpy())
        valid_paths.extend(batch_valid_paths)
    if not all_features:
        return [], np.empty((0, 0), dtype=np.float32)
    return valid_paths, np.concatenate(all_features, axis=0).astype(np.float32)
# ============================================================
# DISTINCTIVENESS WITHIN ONE CATEGORY
# ============================================================
def compute_category_distinctiveness(dataset_name, category_dir):
    image_paths = get_images(category_dir)
    if len(image_paths) < 2:
        print(f"Skipping {dataset_name}/{category_dir.name}: fewer than 2 images.")
        return None
    print(f"Distinctiveness: {dataset_name}/{category_dir.name} ({len(image_paths)} images)")
    image_paths, features = extract_image_features(image_paths)
    n_images = len(image_paths)
    if n_images < 2:
        return None
    similarity_matrix = features @ features.T
    mean_similarity = (
        similarity_matrix.sum(axis=1) - np.diag(similarity_matrix)
    ) / (n_images - 1)
    distinctiveness = 1.0 - mean_similarity
    df = pd.DataFrame({
        "dataset": dataset_name,
        "category": category_dir.name,
        "filename": [path.name for path in image_paths],
        "image_path": [str(path) for path in image_paths],
        "mean_similarity": mean_similarity,
        "distinctiveness": distinctiveness
    })
    df = df.sort_values(
        "distinctiveness",
        ascending=False
    ).reset_index(drop=True)
    df["distinctiveness_rank"] = np.arange(1, len(df) + 1)
    return df
# ============================================================
# SELECT MOST / LEAST DISTINCTIVE
# ============================================================
def select_extremes(df, n=N_SELECT):
    n_images = len(df)
    n_use = min(n, n_images // 2)
    if n_use == 0:
        return pd.DataFrame()
    if n_use < n:
        print(
            f"Warning: {df['dataset'].iloc[0]}/{df['category'].iloc[0]} "
            f"has only {n_images} images after human filtering. "
            f"Selecting {n_use} most and {n_use} least distinctive."
        )
    most = df.head(n_use).copy()
    least = df.tail(n_use).copy()
    most["selection"] = "most_distinctive"
    least["selection"] = "least_distinctive"
    return pd.concat([most, least], ignore_index=True)
# ============================================================
# COPY SELECTED IMAGES
# ============================================================
def copy_selected_images(selected_df):
    for _, row in selected_df.iterrows():
        source = Path(row["image_path"])
        destination = (
            OUTPUT_DIR
            / "selected_images"
            / row["dataset"]
            / row["category"]
            / row["selection"]
        )
        destination.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination / source.name)
# ============================================================
# RUN DISTINCTIVENESS
# ============================================================
def run_distinctiveness():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    ranking_frames = []
    selected_frames = []
    for dataset_name in DATASETS:
        dataset_dir = FILTERED_DIR / dataset_name
        if not dataset_dir.exists():
            print(f"Filtered dataset not found: {dataset_dir}")
            continue
        category_dirs = sorted([
            path for path in dataset_dir.iterdir()
            if path.is_dir()
        ])
        for category_dir in category_dirs:
            df = compute_category_distinctiveness(
                dataset_name,
                category_dir
            )
            if df is None:
                continue
            ranking_frames.append(df)
            selected = select_extremes(df)
            if not selected.empty:
                selected_frames.append(selected)
    if ranking_frames:
        all_rankings = pd.concat(ranking_frames, ignore_index=True)
        all_rankings.to_csv(
            OUTPUT_DIR / "all_image_distinctiveness.csv",
            index=False
        )
    else:
        all_rankings = pd.DataFrame()
    if selected_frames:
        all_selected = pd.concat(selected_frames, ignore_index=True)
        all_selected.to_csv(
            OUTPUT_DIR / "selected_images.csv",
            index=False
        )
        copy_selected_images(all_selected)
    else:
        all_selected = pd.DataFrame()
    print("\nDistinctiveness analysis finished.")
    print(f"Ranking CSV: {OUTPUT_DIR / 'all_image_distinctiveness.csv'}")
    print(f"Selected CSV: {OUTPUT_DIR / 'selected_images.csv'}")
    print(f"Selected images: {OUTPUT_DIR / 'selected_images'}")
    return all_rankings, all_selected
# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 70)
    print("STEP 1: REMOVE PEOPLE AND HUMAN BODY PARTS")
    print("=" * 70)
    run_human_filter()
    print("\n" + "=" * 70)
    print("STEP 2: CALCULATE CLIP-RN50 DISTINCTIVENESS")
    print("=" * 70)
    run_distinctiveness()
    print("\nFinished.")
if __name__ == "__main__":
    main()

Using device: mps


/Users/3447783148qq.com/micromamba/envs/Lab_Project/lib/python3.13/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


OpenCLIP RN50 loaded.
STEP 1: REMOVE PEOPLE AND HUMAN BODY PARTS

Filtering places: 201 categories
places/abbey-300: 300 images
  kept=98, excluded=202
places/airport_terminal-300: 300 images
  kept=117, excluded=183
places/alley-300: 300 images
  kept=158, excluded=142
places/amphitheater-300: 300 images
  kept=164, excluded=136
places/amusement_park-300: 300 images
  kept=57, excluded=243
places/apartment_building-300: 300 images
  kept=202, excluded=98
places/aquarium-300: 300 images
  kept=53, excluded=247
places/aqueduct-300: 300 images
Could not load Places_Konkle_Sample300/places/aqueduct-300/gsun_3eb12e26b495b4b8d2483e9f25609d1c.jpg: cannot identify image file 'Places_Konkle_Sample300/places/aqueduct-300/gsun_3eb12e26b495b4b8d2483e9f25609d1c.jpg'
  kept=92, excluded=207
places/arch-300: 300 images
  kept=108, excluded=192
places/art_gallery-300: 300 images
  kept=80, excluded=220
places/art_studio-300: 300 images
  kept=101, excluded=199
places/assembly_line-300: 300 images
  k

In [2]:
from pathlib import Path
import subprocess

root = Path("/Users/3447783148qq.com/Desktop/Places_Konkle_Sample300")

suspicious_files = [
    path for path in root.rglob("*")
    if path.is_file() and "._" in path.name
]

appledouble = []
real_images = []
other = []

for path in suspicious_files:
    result = subprocess.run(
        ["file", "-b", str(path)],
        capture_output=True,
        text=True
    )

    file_type = result.stdout.strip().lower()

    if "appledouble" in file_type:
        appledouble.append(path)

    elif (
        "jpeg image" in file_type
        or "png image" in file_type
        or "bitmap" in file_type
        or "tiff image" in file_type
        or "webp image" in file_type
    ):
        real_images.append(path)

    else:
        other.append((path, file_type))

print("Actual AppleDouble files:", len(appledouble))
print("Actual image files:", len(real_images))
print("Other/unrecognized files:", len(other))

print("\nExamples of AppleDouble:")
for path in appledouble[:10]:
    print(path)

print("\nExamples of real images:")
for path in real_images[:10]:
    print(path)

print("\nExamples of other files:")
for path, file_type in other[:10]:
    print(path, "->", file_type)

Actual AppleDouble files: 0
Actual image files: 0
Other/unrecognized files: 0

Examples of AppleDouble:

Examples of real images:

Examples of other files:


In [1]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import torch
import open_clip
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# ============================================================
# SETTINGS
# ============================================================
INPUT_DIR = Path("Places_Konkle_Sample300")
FILTERED_DIR = Path("Places_Konkle_No_Humans")
OUTPUT_DIR = Path("clip_rn50_results")
DATASETS = ["places", "konkle"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
N_SELECT = 30
BATCH_SIZE = 32
DINO_BATCH_SIZE = 4

# Lower = more conservative: more detections and therefore more images excluded.
BOX_THRESHOLD = 0.20
TEXT_THRESHOLD = 0.20

# ============================================================
# DEVICE
# ============================================================
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

# ============================================================
# LOAD GROUNDING DINO FOR HUMAN FILTER
# ============================================================
DINO_MODEL_NAME = "IDEA-Research/grounding-dino-base"
dino_processor = AutoProcessor.from_pretrained(DINO_MODEL_NAME)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL_NAME).to(DEVICE)
dino_model.eval()
print("Grounding DINO loaded.")

# ============================================================
# HUMAN / BODY-PART LABELS
# ============================================================
HUMAN_LABELS = [
    "person",
    "human",
    "human face",
    "face",
    "human hand",
    "hand",
    "human arm",
    "arm",
    "human leg",
    "leg",
    "human foot",
    "foot",
    "human body"
]

# ============================================================
# LOAD OPENCLIP RN50 FOR DISTINCTIVENESS
# ============================================================
MODEL_NAME = "RN50"
PRETRAINED = "openai"
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(DEVICE)
model.eval()
print("OpenCLIP RN50 loaded.")

# ============================================================
# IMAGE HELPERS
# ============================================================
def get_images(category_dir):
    return sorted([
        path for path in Path(category_dir).iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ])

def load_image(path):
    with Image.open(path) as img:
        return preprocess(img.convert("RGB"))

def load_pil_image(path):
    with Image.open(path) as img:
        return img.convert("RGB").copy()

# ============================================================
# HUMAN DETECTION
# ============================================================
@torch.no_grad()
def classify_human_batch(image_paths):
    valid_paths = []
    images = []
    for path in image_paths:
        try:
            images.append(load_pil_image(path))
            valid_paths.append(path)
        except Exception as e:
            print(f"Could not load {path}: {e}")
    if not images:
        return []

    text_labels = [HUMAN_LABELS for _ in images]
    inputs = dino_processor(images=images, text=text_labels, return_tensors="pt", padding=True).to(DEVICE)
    outputs = dino_model(**inputs)

    target_sizes = [(img.height, img.width) for img in images]
    detections = dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs["input_ids"],
        threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        target_sizes=target_sizes
    )

    results = []
    for path, detection in zip(valid_paths, detections):
        scores = detection["scores"].detach().cpu().numpy()
        boxes = detection["boxes"].detach().cpu().numpy()
        labels = detection.get("text_labels", detection.get("labels", []))

        if torch.is_tensor(labels):
            labels = labels.detach().cpu().tolist()
        else:
            labels = list(labels)

        contains_human = len(scores) > 0

        if contains_human:
            best_idx = int(np.argmax(scores))
            max_score = float(scores[best_idx])
            strongest_label = str(labels[best_idx]) if best_idx < len(labels) else ""
            strongest_box = boxes[best_idx].tolist()
        else:
            max_score = 0.0
            strongest_label = ""
            strongest_box = []

        results.append({
            "image_path": str(path),
            "num_detections": len(scores),
            "max_detection_score": max_score,
            "strongest_label": strongest_label,
            "strongest_box": strongest_box,
            "contains_human": contains_human
        })
    return results

# ============================================================
# FILTER ONE DATASET
# ============================================================
def filter_dataset(dataset_name):
    source_dataset = INPUT_DIR / dataset_name
    destination_dataset = FILTERED_DIR / dataset_name

    if not source_dataset.exists():
        print(f"Dataset not found: {source_dataset}")
        return pd.DataFrame()

    category_dirs = sorted([
        path for path in source_dataset.iterdir()
        if path.is_dir()
    ])

    records = []
    print(f"\nFiltering {dataset_name}: {len(category_dirs)} categories")

    for category_dir in category_dirs:
        image_paths = get_images(category_dir)
        output_category = destination_dataset / category_dir.name
        output_category.mkdir(parents=True, exist_ok=True)

        kept = 0
        removed = 0

        print(f"{dataset_name}/{category_dir.name}: {len(image_paths)} images")

        for start in range(0, len(image_paths), DINO_BATCH_SIZE):
            batch_paths = image_paths[start:start + DINO_BATCH_SIZE]
            batch_results = classify_human_batch(batch_paths)

            for result in batch_results:
                image_path = Path(result["image_path"])

                if result["contains_human"]:
                    removed += 1
                    decision = "excluded"
                else:
                    shutil.copy2(image_path, output_category / image_path.name)
                    kept += 1
                    decision = "kept"

                records.append({
                    "dataset": dataset_name,
                    "category": category_dir.name,
                    "filename": image_path.name,
                    "image_path": str(image_path),
                    "num_detections": result["num_detections"],
                    "max_detection_score": result["max_detection_score"],
                    "strongest_label": result["strongest_label"],
                    "strongest_box": result["strongest_box"],
                    "decision": decision
                })

        print(f"  kept={kept}, excluded={removed}")

    return pd.DataFrame(records)

# ============================================================
# RUN HUMAN FILTER
# ============================================================
def run_human_filter():
    FILTERED_DIR.mkdir(parents=True, exist_ok=True)
    frames = []

    for dataset in DATASETS:
        df = filter_dataset(dataset)
        if not df.empty:
            frames.append(df)

    if frames:
        results = pd.concat(frames, ignore_index=True)
        results.to_csv(FILTERED_DIR / "human_filter_results.csv", index=False)

        print(f"\nHuman-filter results saved to {FILTERED_DIR / 'human_filter_results.csv'}")
        print(f"Kept: {(results['decision'] == 'kept').sum()}")
        print(f"Excluded: {(results['decision'] == 'excluded').sum()}")

        return results

    return pd.DataFrame()

# ============================================================
# EXTRACT RN50 IMAGE EMBEDDINGS
# ============================================================
@torch.no_grad()
def extract_image_features(image_paths):
    all_features = []
    valid_paths = []

    for start in range(0, len(image_paths), BATCH_SIZE):
        batch_paths = image_paths[start:start + BATCH_SIZE]
        batch_tensors = []
        batch_valid_paths = []

        for path in batch_paths:
            try:
                batch_tensors.append(load_image(path))
                batch_valid_paths.append(path)
            except Exception as e:
                print(f"Could not load {path}: {e}")

        if not batch_tensors:
            continue

        images = torch.stack(batch_tensors).to(DEVICE)
        features = model.encode_image(images)
        features = features / features.norm(dim=-1, keepdim=True)

        all_features.append(features.cpu().numpy())
        valid_paths.extend(batch_valid_paths)

    if not all_features:
        return [], np.empty((0, 0), dtype=np.float32)

    return valid_paths, np.concatenate(all_features, axis=0).astype(np.float32)

# ============================================================
# DISTINCTIVENESS WITHIN ONE CATEGORY
# ============================================================
def compute_category_distinctiveness(dataset_name, category_dir):
    image_paths = get_images(category_dir)

    if len(image_paths) < 2:
        print(f"Skipping {dataset_name}/{category_dir.name}: fewer than 2 images.")
        return None

    print(f"Distinctiveness: {dataset_name}/{category_dir.name} ({len(image_paths)} images)")

    image_paths, features = extract_image_features(image_paths)
    n_images = len(image_paths)

    if n_images < 2:
        return None

    similarity_matrix = features @ features.T
    mean_similarity = (similarity_matrix.sum(axis=1) - np.diag(similarity_matrix)) / (n_images - 1)
    distinctiveness = 1.0 - mean_similarity

    df = pd.DataFrame({
        "dataset": dataset_name,
        "category": category_dir.name,
        "filename": [path.name for path in image_paths],
        "image_path": [str(path) for path in image_paths],
        "mean_similarity": mean_similarity,
        "distinctiveness": distinctiveness
    })

    df = df.sort_values("distinctiveness", ascending=False).reset_index(drop=True)
    df["distinctiveness_rank"] = np.arange(1, len(df) + 1)

    return df

# ============================================================
# SELECT MOST / LEAST DISTINCTIVE
# ============================================================
def select_extremes(df, n=N_SELECT):
    n_images = len(df)
    n_use = min(n, n_images // 2)

    if n_use == 0:
        return pd.DataFrame()

    if n_use < n:
        print(
            f"Warning: {df['dataset'].iloc[0]}/{df['category'].iloc[0]} "
            f"has only {n_images} images after human filtering. "
            f"Selecting {n_use} most and {n_use} least distinctive."
        )

    most = df.head(n_use).copy()
    least = df.tail(n_use).copy()

    most["selection"] = "most_distinctive"
    least["selection"] = "least_distinctive"

    return pd.concat([most, least], ignore_index=True)

# ============================================================
# COPY SELECTED IMAGES
# ============================================================
def copy_selected_images(selected_df):
    for _, row in selected_df.iterrows():
        source = Path(row["image_path"])
        destination = OUTPUT_DIR / "selected_images" / row["dataset"] / row["category"] / row["selection"]
        destination.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination / source.name)

# ============================================================
# RUN DISTINCTIVENESS
# ============================================================
def run_distinctiveness():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    ranking_frames = []
    selected_frames = []

    for dataset_name in DATASETS:
        dataset_dir = FILTERED_DIR / dataset_name

        if not dataset_dir.exists():
            print(f"Filtered dataset not found: {dataset_dir}")
            continue

        category_dirs = sorted([
            path for path in dataset_dir.iterdir()
            if path.is_dir()
        ])

        for category_dir in category_dirs:
            df = compute_category_distinctiveness(dataset_name, category_dir)

            if df is None:
                continue

            ranking_frames.append(df)

            selected = select_extremes(df)

            if not selected.empty:
                selected_frames.append(selected)

    if ranking_frames:
        all_rankings = pd.concat(ranking_frames, ignore_index=True)
        all_rankings.to_csv(OUTPUT_DIR / "all_image_distinctiveness.csv", index=False)
    else:
        all_rankings = pd.DataFrame()

    if selected_frames:
        all_selected = pd.concat(selected_frames, ignore_index=True)
        all_selected.to_csv(OUTPUT_DIR / "selected_images.csv", index=False)
        copy_selected_images(all_selected)
    else:
        all_selected = pd.DataFrame()

    print("\nDistinctiveness analysis finished.")
    print(f"Ranking CSV: {OUTPUT_DIR / 'all_image_distinctiveness.csv'}")
    print(f"Selected CSV: {OUTPUT_DIR / 'selected_images.csv'}")
    print(f"Selected images: {OUTPUT_DIR / 'selected_images'}")

    return all_rankings, all_selected

# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 70)
    print("STEP 1: REMOVE PEOPLE AND HUMAN BODY PARTS WITH GROUNDING DINO")
    print("=" * 70)

    run_human_filter()

    print("\n" + "=" * 70)
    print("STEP 2: CALCULATE CLIP-RN50 DISTINCTIVENESS")
    print("=" * 70)

    run_distinctiveness()

    print("\nFinished.")

if __name__ == "__main__":
    main()

Using device: mps


Loading weights:   0%|          | 0/1182 [00:00<?, ?it/s]

Grounding DINO loaded.


/Users/3447783148qq.com/micromamba/envs/Lab_Project/lib/python3.13/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


OpenCLIP RN50 loaded.
STEP 1: REMOVE PEOPLE AND HUMAN BODY PARTS WITH GROUNDING DINO

Filtering places: 201 categories
places/abbey-300: 300 images


/Users/3447783148qq.com/micromamba/envs/Lab_Project/lib/python3.13/site-packages/transformers/models/grounding_dino/processing_grounding_dino.py:94: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)


  kept=226, excluded=74
places/airport_terminal-300: 300 images
  kept=50, excluded=250
places/alley-300: 300 images
  kept=127, excluded=173
places/amphitheater-300: 300 images
  kept=123, excluded=177
places/amusement_park-300: 300 images
  kept=97, excluded=203
places/apartment_building-300: 300 images
  kept=210, excluded=90
places/aquarium-300: 300 images
  kept=182, excluded=118
places/aqueduct-300: 300 images
Could not load Places_Konkle_Sample300/places/aqueduct-300/gsun_3eb12e26b495b4b8d2483e9f25609d1c.jpg: cannot identify image file 'Places_Konkle_Sample300/places/aqueduct-300/gsun_3eb12e26b495b4b8d2483e9f25609d1c.jpg'
  kept=220, excluded=79
places/arch-300: 300 images
  kept=200, excluded=100
places/art_gallery-300: 300 images
  kept=37, excluded=263
places/art_studio-300: 300 images
  kept=50, excluded=250
places/assembly_line-300: 300 images
  kept=84, excluded=216
places/attic-300: 300 images
  kept=214, excluded=86
places/auditorium-300: 300 images
  kept=75, excluded=2